## Bivariate Analysis: Feature-wise Association with Targets (Has a high blood pressure, Has diabetes & Cardiovascular condition)

In this step, we explore the relationship between each categorical feature and the chosen target variable using:

- **Chi-square test of independence** to detect statistical association.
- **Normalized stacked bar plots** to visualize category-wise distribution with respect to the target.

The output includes:
- A folder with visual plots (`bivariate_plots_<target>`)
- A CSV file containing chi-square p-values (`bivariate_chi_square_vs_<target>.csv`)

This helps identify features with strong association for downstream modeling and interpretation.


In [1]:
import os
import pandas as pd

# --- Path Config ---
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")
STATS_DIR = os.path.join(OUTPUTS_DIR, "statistics")
PLOTS_DIR = os.path.join(OUTPUTS_DIR, "plots")
METRICS_DIR = os.path.join(OUTPUTS_DIR, "metrics")

# Make sure subfolders exist
os.makedirs(STATS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)


In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import os
import re

# Load the final cleaned dataset
df = pd.read_csv(os.path.join(STATS_DIR, "final_cleaned_data.csv"))

# Utility function to clean file names safely
def clean_filename(s):
    return re.sub(r'[^\w\-_.]', '_', s)[:100]

# Define the reusable function
def run_bivariate_analysis(df, target):
    print(f" Running bivariate analysis for: {target}")
    
    # Create subdirectory for plots (bivariate analysis)
    part7_plots_dir = os.path.join(PLOTS_DIR, "part7_bivariate_analysis")
    os.makedirs(part7_plots_dir, exist_ok=True)

    # Store chi-square results
    chi_results = []

    for col in df.columns:
        if col == target:
            continue

        ct = pd.crosstab(df[col], df[target])

        try:
            chi2, p, dof, expected = chi2_contingency(ct)
            chi_results.append({"Feature": col, "p-value": round(p, 4)})
        except:
            chi_results.append({"Feature": col, "p-value": "Error"})

        # Plot normalized stacked bar chart
        ct_norm = ct.div(ct.sum(axis=1), axis=0)

        proportions = df[col].value_counts(normalize=True).sort_index() * 100
        new_labels = [f"{label} ({proportions[label]:.1f}%)" if label in proportions else str(label) for label in ct.index]
        ct_norm.index = new_labels

        ct_norm.plot(kind='bar', stacked=True, figsize=(9, 5), colormap="Pastel1")
        plt.title(f"{col} vs {target}")
        plt.ylabel("Proportion")
        plt.xlabel(f"{col} (with % share in population)")
        plt.tight_layout()

        # Save plot
        plt.savefig(os.path.join(part7_plots_dir, f"{clean_filename(col)}_vs_{clean_filename(target)}.png"))
        plt.close()

    # Save chi-square results
    chi_df = pd.DataFrame(chi_results)
    # Save chi-square results into metrics
    chi_df.to_csv(os.path.join(METRICS_DIR, f"bivariate_chi_square_vs_{clean_filename(target)}.csv"), index=False)

    print(f" Completed: {target} → Plots saved in '{part7_plots_dir}', chi-square results in '{METRICS_DIR}'\n")


# Run for all three targets
targets = [
    "Has a high blood pressure",
    "Has diabetes",
    "Cardiovascular condition (Heart disease or stroke)"
]

for target in targets:
    run_bivariate_analysis(df, target)


 Running bivariate analysis for: Has a high blood pressure


C:\Users\saini\AppData\Local\Temp\ipykernel_26916\3657168168.py:49: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


 Completed: Has a high blood pressure → Plots saved in 'd:\Projects\health-risk-prediction\outputs\plots\part7_bivariate_analysis', chi-square results in 'd:\Projects\health-risk-prediction\outputs\metrics'

 Running bivariate analysis for: Has diabetes


C:\Users\saini\AppData\Local\Temp\ipykernel_26916\3657168168.py:49: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


 Completed: Has diabetes → Plots saved in 'd:\Projects\health-risk-prediction\outputs\plots\part7_bivariate_analysis', chi-square results in 'd:\Projects\health-risk-prediction\outputs\metrics'

 Running bivariate analysis for: Cardiovascular condition (Heart disease or stroke)


C:\Users\saini\AppData\Local\Temp\ipykernel_26916\3657168168.py:49: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


 Completed: Cardiovascular condition (Heart disease or stroke) → Plots saved in 'd:\Projects\health-risk-prediction\outputs\plots\part7_bivariate_analysis', chi-square results in 'd:\Projects\health-risk-prediction\outputs\metrics'



## Bivariate Analysis Summary & Feature Selection Plan

- Conducted bivariate EDA (chi-square + visual plots) for all three target variables:
  - **High Blood Pressure**
  - **Diabetes**
  - **Cardiovascular Condition (Heart disease or stroke)**

### Key Findings:
- **Almost all features show strong statistical association** with the targets (p-value ≈ 0).
- For **High Blood Pressure**, only three features showed weaker association:
  - `Sex at Birth` (p = 0.106)
  - `Mood disorder` (p = 0.0132)
  - `Anxiety disorder` (p = 0.0126)
- Despite weaker p-values, these features remain **clinically important** and interpretable.

### Decision:
- **No features will be dropped** at this stage.
- All variables will be retained for multivariate modeling.
